# W-BOT v5 — Qwen3-4B Fine-Tune (Colab A100)

**Dataset:** wbot_v5_train.jsonl (3787 kayıt)
**Base model:** Qwen/Qwen3-4B
**Platform:** Google Colab A100-SXM4-40GB
**Çıktı:** Qwen3-4B-wbot_v5-Q4_K_M.gguf

wbot_v4'ten fark: +182 yeni kayıt (C paketi — `scripts/rebuild_wbot_v5_train.py`)
  - S34/V02 modifikasyon sipariş sonrası (20)
  - V04 küfür genişletme (18)
  - V06/E27 alerji kalıp düzeltmesi (24)
  - S41/V07 eskalasyon (20)
  - Anti-hallüsinasyon (100)

Ayrıca `_SYSTEM_TEMPLATE` (inference, `llama_cpp_backend.py`/`qwen3_backend.py`)
W11 kapanış kuralı revizyonu aldı — bu, training pipeline'ını ETKİLEMEZ
(`short_prompt=True` varsayılanında dataset'in `system` alanı yok sayılıp
`_SYSTEM_SHORT` kullanılıyor, bkz. PROJE_DURUMU.md görev #29). Model,
hiperparametreler ve GGUF dönüşüm adımları wbot_v4 ile aynı — tek fark
dataset. `--full-prompt` KULLANILMIYOR (bilerek — gerekçe: görev #29,
`max_seq_len=800` tam şablonu barındıramıyor). Eğitim scripti
(`train_wbot_v2.py`) repodan olduğu gibi çağrılır, notebook içine
kopyalanmaz.

## 2. GPU Kontrolü

Runtime > Change runtime type > A100 GPU seçili olmalı.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "GPU bulunamadı!"

## 3. Bağımlılık Kurulumu

`robot_waiter_ai/training/requirements_train.txt` ile birebir aynı paket ve
versiyon pinleri (repo henüz klonlanmadığı için burada elle yazıldı —
transformers>=4.43.0 Qwen3 chat template desteği için zorunlu).

In [ ]:
!pip install -q "transformers>=4.43.0" "datasets>=2.20.0" "bitsandbytes>=0.46.1" \
    "peft>=0.10.0" "accelerate>=0.27.0"

## 4. Google Drive Mount

`/content/drive/MyDrive/wbot/` altında dataset ve çıktıların bulunması
beklenir. Kendi Drive yapınıza göre aşağıdaki path'leri güncelleyin.
`wbot_v5_train.jsonl` henüz repoya commit edilmediği için buraya elle
yüklemeniz gerekiyor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Repo Klonlama

`train_wbot_v2.py` VE `wbot_v5_train.jsonl` (3787 kayıt, commit `58d3ec8`
ile repoya eklendi) repodan klonlanarak gelir — dataset'i ayrıca Drive'a
yüklemenize gerek YOK.

**Repo public** (doğrulandı) — `GITHUB_TOKEN` secret'ı gerekmez, aşağıdaki
hücre token'sız clone yapar. Repo ileride private'a alınırsa: Colab sol
panelden 🔑 **Secrets** sekmesine gidip `GITHUB_TOKEN` adında bir secret
ekleyin (GitHub → Settings → Developer settings → Personal access tokens
→ en az `repo` yetkili bir token).


In [ ]:
import os

REPO_DIR = "/content/Garson-bot"
REPO_URL = "https://github.com/MustafaEmreBiyik/Garson-bot.git"

# Repo private ise Colab Secrets'tan GITHUB_TOKEN okunur (yoksa token'sız denenir)
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None

clone_url = REPO_URL if not _token else REPO_URL.replace("https://", f"https://{_token}@")

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
    # Alternatif — repoyu elle Drive'a yükleyip buradan kopyalamak isterseniz:
    # !cp -r /content/drive/MyDrive/wbot/Garson-bot {REPO_DIR}
else:
    print("Repo zaten mevcut:", REPO_DIR)

%cd {REPO_DIR}

## 6. Dataset Kontrolü

Dataset klonlanan repo içinde geldi (bkz. Hücre 5) — burada yalnızca
kayıt sayısını doğruluyoruz.


In [ ]:
# Dataset path — repodan klonlandı (Hücre 5), Drive'a ayrıca yüklemeye gerek yok
DATASET_PATH = f"{REPO_DIR}/robot_waiter_ai/datasets/processed/wbot_v5_train.jsonl"
OUTPUT_DIR   = "/content/drive/MyDrive/wbot/wbot_v5_output"
MODEL_NAME   = "Qwen/Qwen3-4B"

# Kontrol
import os
assert os.path.exists(DATASET_PATH), f"Dataset bulunamadı: {DATASET_PATH}"
import subprocess
line_count = int(subprocess.check_output(["wc", "-l", DATASET_PATH]).split()[0])
print(f"Dataset: {line_count} kayıt (beklenen: 3787)")
assert line_count == 3787, f"Beklenmeyen kayıt sayısı: {line_count}"


## 7. Eğitim

`train_wbot_v2.py` repodan klonlanıp olduğu gibi çağrılır (script notebook
içine kopyalanmaz). Gerçek CLI argümanları (`train_wbot_v2.py` argparse
tanımından — tire ile, alt çizgi değil): `--dataset`, `--output-dir`,
`--epochs`, `--run-eval`.


In [ ]:
# --drive-dir kullanılmadı: OUTPUT_DIR zaten Drive path'inde (Hücre 5),
# checkpoint'ler eğitim sırasında doğrudan Drive'a yazılır.
# --resume auto: OUTPUT_DIR'daki en son checkpoint varsa oradan devam eder,
# yoksa sıfırdan başlar (train_wbot_v2.py --resume desteği, satır ~429-433).
!python robot_waiter_ai/training/train_wbot_v2.py \
    --dataset "{DATASET_PATH}" \
    --output-dir "{OUTPUT_DIR}" \
    --epochs 3 \
    --resume auto \
    --run-eval


## 8. GGUF Dönüşümü

PROJE_DURUMU.md → "GGUF Dönüşümü — Colab Hücreleri" ile birebir aynı akış,
dosya adları wbot_v5 olacak şekilde güncellendi.

In [ ]:
# Kurulum
!pip install -q transformers peft accelerate
!apt-get install -q -y build-essential cmake
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j4

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
MERGED_DIR  = "/content/wbot_v5_merged"
GGUF_PATH   = "/content/drive/MyDrive/wbot/Qwen3-4B-wbot_v5-Q4_K_M.gguf"

In [ ]:
# Merge (adapter + base model)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Base model yükleniyor...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="cpu"
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print("Merge ediliyor...")
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained(ADAPTER_DIR).save_pretrained(MERGED_DIR)
print("Merge tamam:", MERGED_DIR)

In [ ]:
# GGUF dönüşümü
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outtype q4_k_m \
    --outfile {GGUF_PATH}
print("GGUF kaydedildi:", GGUF_PATH)

## 9. Drive'a Kaydet

`GGUF_PATH` (Hücre 7) zaten doğrudan Drive'a yazıldı — bu hücre yalnızca
dosyanın var olduğunu doğrular. `GGUF_PATH` lokal (`/content/...`) bir yola
ayarlanmışsa aşağıdaki yorum satırını açarak Drive'a kopyalayın.

In [ ]:
import os
import shutil

GGUF_OUTPUT = "/content/drive/MyDrive/wbot/Qwen3-4B-wbot_v5-Q4_K_M.gguf"

# GGUF_PATH lokal bir yoldaysa (Drive değilse) önce buraya kopyalayın:
# shutil.copy(GGUF_PATH, GGUF_OUTPUT)

assert os.path.exists(GGUF_OUTPUT), f"GGUF bulunamadı: {GGUF_OUTPUT}"
size_mb = os.path.getsize(GGUF_OUTPUT) / 1e6
print(f"✓ GGUF kaydedildi: {GGUF_OUTPUT} ({size_mb:.0f} MB)")

## Eğitim Tamamlandı

Sonraki adımlar:
1. GGUF dosyasını Jetson'a kopyala: `/home/emk/models/`
2. `demo_usb.py`'deki model path'i güncelle
3. Eval: `python3 scripts/eval_gguf.py` → hedef 31/32 (bkz. PROJE_DURUMU.md
   görev #29 beklenti tablosu; E19/E16 bilinen, prompt-kaynaklı, retrain'le
   düzelmeyecek bir regresyon olarak kalabilir — ayrı görev)
4. V01-V07: `python3 scripts/eval_gguf.py --v4-targets` → hedef 37/39